# Logging - Shared Configuration and Helper Functions

This notebook holds everything that is **shared across the pipeline**: configuration values (table names, thresholds, paths) and small reusable functions (loading CSVs, saving Delta tables, converting dates, validating row counts).

We use Python's built-in `logging` module instead of plain `print()` statements. This gives every message a timestamp and a severity level (INFO, WARNING, ERROR), which is standard practice in real data pipelines.

This keeps the project **modular** (reusable functions instead of copy-pasted code) and **parameterized** (thresholds and table names live in one place, not hardcoded everywhere).

### Step 1: Set Up Logging


In [0]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True  # ensures logging works cleanly even if Databricks pre-configures logging
)
logger = logging.getLogger("ServiceTrack")

logger.info("Logger initialized for ServiceTrack pipeline.")

2026-07-11 20:09:02,299 - INFO - Logger initialized for ServiceTrack pipeline.


### Step 2: Configuration Values (Parameterized Settings)
.

In [0]:
# Create a Databricks widget for the base data folder path (only needs to run once per notebook session).
# dbutils.widgets.get() will read whatever value is currently set in the widget box at the top of the notebook.
dbutils.widgets.text("volume_base_path", "/Volumes/workspace/default/project_dataset", "Raw Data Folder Path")

#File paths (now parameterized via widget instead of hardcoded)
VOLUME_BASE_PATH = dbutils.widgets.get("volume_base_path")
CUSTOMERS_CSV_PATH = f"{VOLUME_BASE_PATH}/customers.csv"
DEVICES_CSV_PATH = f"{VOLUME_BASE_PATH}/devices.csv"
SERVICE_JOBS_CSV_PATH = f"{VOLUME_BASE_PATH}/service_jobs.csv"

#Bronze table names
BRONZE_CUSTOMERS_TABLE = "bronze_customers"
BRONZE_DEVICES_TABLE = "bronze_devices"
BRONZE_SERVICE_JOBS_TABLE = "bronze_service_jobs"

#Silver table name
SILVER_ENRICHED_JOBS_TABLE = "silver_enriched_jobs"

#Gold table names
GOLD_TECHNICIAN_PERFORMANCE_TABLE = "gold_technician_performance"
GOLD_DELAY_ANALYSIS_TABLE = "gold_delay_analysis"
GOLD_REPEAT_CUSTOMERS_TABLE = "gold_repeat_customers"
GOLD_REPEAT_CUSTOMERS_OVERALL_TABLE = "gold_repeat_customers_overall"
GOLD_DEVICE_BRAND_ANALYSIS_TABLE = "gold_device_brand_analysis"
GOLD_CUSTOMER_LATEST_VISIT_TABLE = "gold_customer_latest_visit"

#Business thresholds (parameterized, not hardcoded in logic)
DELAY_RATE_THRESHOLD_PCT = 10.0        
REPEAT_VISIT_MIN_TOTAL = 3          

#Expected row counts (used for pipeline validation)
EXPECTED_BRONZE_JOB_COUNT = 1510
EXPECTED_SILVER_JOB_COUNT = 1500

logger.info(f"Configuration values loaded. Using data folder: {VOLUME_BASE_PATH}")

2026-07-11 20:09:31,576 - INFO - Configuration values loaded. Using data folder: /Volumes/workspace/default/project_dataset


### Step 3: Reusable Function - Load a CSV File


In [0]:
def load_csv(path: str):
    """
    Read a CSV file into a PySpark DataFrame with header and inferred schema.
    Logs success or failure, and raises the error after logging so the notebook still stops
    if something is genuinely wrong (we don't want to silently continue with missing data).
    """
    try:
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(path)
        logger.info(f"Successfully loaded CSV: {path} ({df.count()} rows)")
        return df
    except Exception as e:
        logger.error(f"Failed to load CSV at {path}: {e}")
        raise

### Step 4: Reusable Function - Save a DataFrame as a Delta Table

In [0]:
def save_as_delta(df, table_name: str, overwrite_schema: bool = False):
    """
    Save a DataFrame as a Delta table using overwrite mode.
    Set overwrite_schema=True when the table's column structure has changed
    (e.g. Silver layer after adding new columns).
    """
    try:
        writer = df.write.format("delta").mode("overwrite")
        if overwrite_schema:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(table_name)
        logger.info(f"Saved Delta table: {table_name}")
    except Exception as e:
        logger.error(f"Failed to save Delta table {table_name}: {e}")
        raise

### Step 5: Reusable Function - Convert Multiple Columns to Date Type


In [0]:
from pyspark.sql.functions import col, to_date

def convert_dates(df, columns: list, date_format: str = "yyyy-MM-dd"):
    """
    Convert a list of column names in a DataFrame to proper Date type.
    Logs a warning if a column ends up fully null after conversion,
    since that usually means the date format did not match the raw data.
    """
    for column_name in columns:
        df = df.withColumn(column_name, to_date(col(column_name), date_format))
    logger.info(f"Converted columns to Date type: {columns}")
    return df

### Step 6: Reusable Function - Validate Row Counts


In [0]:
def validate_row_count(actual_count: int, expected_count: int, label: str):
    """
    Compare an actual row count against an expected value and log a Success or Warning message.
    Returns True if the check passed, False otherwise (so calling code can react if needed).
    """
    if actual_count == expected_count:
        logger.info(f"Success: {label} row count matched expected value ({actual_count}).")
        return True
    else:
        logger.warning(f"Warning: {label} row count was {actual_count}, expected {expected_count}.")
        return False

### Step 7: Reusable Function - Validate No Negative Costs


In [0]:
from pyspark.sql.functions import col as _col

def validate_no_negative_costs(df, estimated_col: str = "estimated_cost", actual_col: str = "actual_cost"):
    """
    Check that estimated_cost and actual_cost are never negative in the given DataFrame.
    Logs a Success message if none are found, or a Warning with the count if any are found.
    Returns True if the check passed, False otherwise.
    """
    negative_estimated = df.filter(_col(estimated_col) < 0).count()
    negative_actual = df.filter(_col(actual_col) < 0).count()

    if negative_estimated == 0 and negative_actual == 0:
        logger.info("Success: No negative values found in estimated_cost or actual_cost.")
        return True
    else:
        logger.warning(
            f"Warning: Found {negative_estimated} negative estimated_cost rows "
            f"and {negative_actual} negative actual_cost rows."
        )
        return False

Any notebook that starts with `%run ./00_Config_Utils` now has access to:
- `logger` for logging
- All configuration values (table names, thresholds, paths — including the parameterized data folder path)
- `load_csv()`, `save_as_delta()`, `convert_dates()`, `validate_row_count()`, `validate_no_negative_costs()`